# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library, referencing all dataset entities by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # Suppress pandas warnings for a clearer notebook

# Define the Croissant schema URL for this dataset
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print basic dataset information
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else 'N/A'}")

## 2. Data Overview
The Croissant schema for this dataset may contain multiple record sets, each with its own fields/columns.

The `mlcroissant` interface allows you to inspect all available record sets, along with their field `@id`s.

**Below, we enumerate the available record sets and their fields, referencing entities by their `@id`.**

In [ ]:
# Display available record sets and their fields with @id references
print('Available Record Sets:')
for record_set in metadata.record_sets:
    print(f"  Record Set @id: {record_set.id}")
    print(f"    Name:     {record_set.name}")
    print(f"    Fields (@id):")
    for field in record_set.fields:
        print(f"      - {field.id} (name: {field.name}, type: {field.data_type})")
    print('')

## 3. Data Extraction
We'll extract data from the primary record set(s) into pandas DataFrames for analysis.

**Note:** Replace `<record_set_id>` and `<field_id>` with actual `@id`s as obtained above. For demonstration, we'll select the first record set for detailed exploration.

In [ ]:
# Gather all Record Set @id values
record_set_ids = [rec.id for rec in metadata.record_sets]
# Display for user reference
print('Record Sets in dataset:')
for rid in record_set_ids:
    print(' -', rid)

# Choose the first record set for detailed analysis (can customize as needed)
main_record_set_id = record_set_ids[0] if record_set_ids else None
print(f'\nSelected for analysis: {main_record_set_id}')

dataframes = {}
for record_set_id in record_set_ids:
    # Extract records from each record set as a list of dicts
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'Loaded {len(df)} records for {record_set_id}')

# Examine available columns in the main record set
print(f"\nColumns in main record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the data using typical EDA techniques such as:
- Filtering records by value
- Normalizing numeric fields
- Grouping and aggregating data

All field references are made by their `@id`, following the Croissant ontology and the dataset specification.

In [ ]:
# You may need to consult field names or `@id`s from Section 2 to pick numeric and group fields.
# For demonstration, let's select GAD-7 total score if present, or the first numeric field available.

main_df = dataframes[main_record_set_id].copy()
# Print available columns for reference
print('Available columns:', main_df.columns.tolist())

# Try to find a likely numeric field: look for names containing 'gad', 'phq', or 'score'. Otherwise, use the first float/integer field from the schema.
field_candidates = []
for rec_set in metadata.record_sets:
    if rec_set.id == main_record_set_id:
        for field in rec_set.fields:
            if field.data_type in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']:
                field_candidates.append((field.id, field.name))

numeric_field_id = None
for f_id, f_name in field_candidates:
    fname = str(f_id).lower() + ' ' + str(f_name).lower()
    if 'gad' in fname or 'phq' in fname or 'score' in fname:
        numeric_field_id = f_id if f_id in main_df.columns else f_name if f_name in main_df.columns else None
        if numeric_field_id:
            break
# Fall back to first numeric field
if not numeric_field_id and field_candidates:
    numeric_field_id = field_candidates[0][0] if field_candidates[0][0] in main_df.columns else field_candidates[0][1]

print(f"Selected numeric field for EDA: {numeric_field_id}\n")

# Drop missing values for numeric analysis
numeric_series = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
main_df = main_df[numeric_field_id].dropna().to_frame()
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

threshold = 5  # You can adjust this threshold for your analysis
filtered_df = dataframes[main_record_set_id][pd.to_numeric(dataframes[main_record_set_id][numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field for filtered records (z-score)
col_mean = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
col_std = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
filtered_df[f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - col_mean
) / (col_std if col_std else 1)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# For grouping, pick a likely categorical field (e.g., gender, education) by inspecting field names
categorical_candidates = []
for rec_set in metadata.record_sets:
    if rec_set.id == main_record_set_id:
        for field in rec_set.fields:
            t = str(field.data_type).lower()
            if t in ['text', 'string', 'schema:text', 'schema:string', 'category'] and field.id in filtered_df.columns:
                categorical_candidates.append(field.id)
# Select first available group field
group_field_id = None
for f_id in categorical_candidates:
    if 'gender' in f_id.lower():
        group_field_id = f_id
        break
if not group_field_id and categorical_candidates:
    group_field_id = categorical_candidates[0]

# Group and show aggregate if possible
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id]
        .mean()
        .reset_index()
        .rename(columns={numeric_field_id: f"{numeric_field_id}_mean"})
    )
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable categorical group-by field found for this record set.")

## 5. Visualization
Let's create basic visualizations to explore the distribution of the selected numeric field and, if possible, compare distributions across a categorical variable.


In [ ]:
# Distribution plot for numeric field
fig, ax = plt.subplots(figsize=(7, 4))
main_data = pd.to_numeric(dataframes[main_record_set_id][numeric_field_id], errors='coerce').dropna()
ax.hist(main_data, bins=15, color='skyblue', edgecolor='k')
ax.set_title(f'Distribution of {numeric_field_id}')
ax.set_xlabel(numeric_field_id)
ax.set_ylabel('Frequency')
plt.show()

# Grouped boxplot by category if possible
if group_field_id and group_field_id in dataframes[main_record_set_id].columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    box_data = dataframes[main_record_set_id][[group_field_id, numeric_field_id]].dropna()
    box_data[numeric_field_id] = pd.to_numeric(box_data[numeric_field_id], errors='coerce')
    box_data = box_data.dropna()
    box_data.boxplot(column=numeric_field_id, by=group_field_id, ax=ax,
                     grid=False, patch_artist=True, boxprops=dict(facecolor='lightgrey'))
    ax.set_title(f'{numeric_field_id} by {group_field_id}')
    plt.suptitle("")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.show()
else:
    print("No suitable group-by categorical field found for comparison plot.")

## 6. Conclusion
In this notebook, we've demonstrated how to explore the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

- We loaded the Croissant schema and referenced all entities using their `@id` fields.
- We listed available record sets and fields, extracted data for analysis, and conducted filtering, normalization, and grouping.
- We visualized the distribution of a numeric mental health measure and, where possible, compared it across demographic groups.

This workflow provides a reproducible and FAIR path for dataset exploration, ensuring clarity around data origin and structure for all downstream analysis.
